In [4]:
import pandas as pd
import numpy as np

iter_df = pd.read_csv('../results/statistical_tests/ms_distributions.csv')

def get_pairwise_pvalue(df, metric, model_a, model_b):
    """Computes the empirical bootstrap p-value exactly as implemented in the pipeline."""
    series_a = df[df['method'] == model_a].set_index('bootstrap_iter')[metric]
    series_b = df[df['method'] == model_b].set_index('bootstrap_iter')[metric]
    
    common = series_a.dropna().index.intersection(series_b.dropna().index)
    diff = series_a.loc[common].values - series_b.loc[common].values
    
    # Exact logic from compute_ci_and_tests
    prop_same_sign = (diff > 0).mean() if diff.mean() > 0 else (diff < 0).mean()
    pval = 2 * min(prop_same_sign, 1 - prop_same_sign)
    
    return pval, np.mean(series_a.loc[common]), np.mean(series_b.loc[common])

# Test Random+CCR vs Global H_tau exactly as the reviewer did
pval_htau, mean_rand, mean_htau = get_pairwise_pvalue(
    iter_df, 
    metric='ccaugrc_progressor', 
    model_a='random_ccr', 
    model_b='h_tau'
)

print(f"Random+CCR mean: {mean_rand:.4f} | Global H_tau mean: {mean_htau:.4f}")
print(f"Empirical p-value: {pval_htau:.3f}")

Random+CCR mean: 0.1605 | Global H_tau mean: 0.2178
Empirical p-value: 0.004


In [ ]:
import pandas as pd
import numpy as np


# 1. Load distributions
ms_iter_df = pd.read_csv('../results/statistical_tests/ms_distributions.csv')
ad_iter_df = pd.read_csv('../results/statistical_tests/ad_distributions.csv')

print("--- MS COHORT CLAIMS ---")
# Random+CCR vs H_tau+CCR on progressor ccAUGRC
pval_1, mean_rand, mean_prop = get_pairwise_pvalue(
    ms_iter_df, 'ccaugrc_progressor', 'random_ccr', 'h_tau_ccr'
)
print(f"Random+CCR vs H_tau+CCR  | Target: p<0.001 | Actual: {pval_1:.4f}")

#H_Total+CCR vs H_tau+CCR on progressor ccAUGRC
pval_2, mean_htot_ccr, _ = get_pairwise_pvalue(
    ms_iter_df, 'ccaugrc_progressor', 'h_total_ccr', 'h_tau_ccr'
)
print(f"H_Total+CCR vs H_tau+CCR | Target: p<0.01  | Actual: {pval_2:.4f}")

print("\n--- AD COHORT CLAIMS ---")
#H_Total vs H_tau+CCR on progressor ccAUGRC
pval_3, mean_htot_prog, mean_prop_prog = get_pairwise_pvalue(
    ad_iter_df, 'ccaugrc_progressor', 'h_total', 'h_tau_ccr'
)
print(f"H_Total vs H_tau+CCR (Progressor) | Target: p<0.001 | Actual: {pval_3:.4f}")

# H_Total vs H_tau+CCR on stable ccAUGRC
pval_4, mean_htot_stab, mean_prop_stab = get_pairwise_pvalue(
    ad_iter_df, 'ccaugrc_stable', 'h_total', 'h_tau_ccr'
)
print(f"H_Total vs H_tau+CCR (Stable)     | Target: p<0.001 | Actual: {pval_4:.4f}")

--- MS COHORT CLAIMS ---
Random+CCR vs H_tau+CCR  | Target: p<0.001 | Actual: 0.0000
H_Total+CCR vs H_tau+CCR | Target: p<0.01  | Actual: 0.0080

--- AD COHORT CLAIMS ---
H_Total vs H_tau+CCR (Progressor) | Target: p<0.001 | Actual: 0.0000
H_Total vs H_tau+CCR (Stable)     | Target: p<0.001 | Actual: 0.0000
